**#Step 1. Methagenom annotation#**

After the preliminary stage, we begin to prepare tools for annotation of metagenomes.

**Substep 1.1** Installing and preparing databases

In [ ]:
# Kraken2: PlusPFP database (about 170 GB). Downloaded and unpacked in /mnt/tank/scratch/ris/databases/kraken2/
cd /mnt/tank/scratch/ris/databases/kraken2
wget ... # link to the archive
tar -xzvf k2_pluspfp_20251015.tar.gz   # unpacking (took ~1-2 hours)

MetaPhlAn4: база vJan25 в /mnt/tank/scratch/ris/databases/metaphlan/. # The Bowtie2 index files are in place

**Substep 1.2.1** Test run of Kraken2 on a single sample

run_kraken_one.sbatch script (4 threads, 250 GB of memory).

In [ ]:
#!/bin/bash
#SBATCH -J kraken_one
#SBATCH -n 1
#SBATCH --cpus-per-task=4
#SBATCH --mem=250G
#SBATCH -t 04:00:00
#SBATCH --partition=main
#SBATCH --error=/mnt/tank/scratch/ris/SRR_files/Logs/kraken_one_%j.err
#SBATCH --output=/mnt/tank/scratch/ris/SRR_files/Logs/kraken_one_%j.out

source /nfs/home/ris/miniforge3/etc/profile.d/mamba.sh
mamba activate snakemake

cd /mnt/tank/scratch/ris/SRR_files
KRAKEN_DB="/mnt/tank/scratch/ris/databases/kraken2"
sample=$(ls *_1_trimmed_paired.fastq.gz | head -1 | sed 's/_1_trimmed_paired.fastq.gz//')
kraken2 --db "$KRAKEN_DB" --threads $SLURM_CPUS_PER_TASK --paired \
  ${sample}_1_trimmed_paired.fastq.gz ${sample}_2_trimmed_paired.fastq.gz \
  --output ${sample}_kraken_test.out --report ${sample}_kraken_test.report --use-names
  
# Test is successful: time ~22 min, memory ~228 GB

**Substep 1.2.2** Test run of MetaPhlAn on a single sample

run_metaphls_one.sbatch script (8 threads, 60 GB of memory). 

*Important: the --mapout parameter is needed for paired ends.*

In [ ]:
#!/bin/bash
#SBATCH -J metaphlan_one
#SBATCH -n 1
#SBATCH --cpus-per-task=8
#SBATCH --mem=60G
#SBATCH -t 04:00:00
#SBATCH --partition=main
#SBATCH --error=/mnt/tank/scratch/ris/SRR_files/Logs/metaphlan_one_%j.err
#SBATCH --output=/mnt/tank/scratch/ris/SRR_files/Logs/metaphlan_one_%j.out

source /nfs/home/ris/miniforge3/etc/profile.d/mamba.sh
mamba activate snakemake

cd /mnt/tank/scratch/ris/SRR_files
METAPHLAN_DB="/mnt/tank/scratch/ris/databases/metaphlan"
METAPHLAN_INDEX="mpa_vJan25_CHOCOPhlAnSGB_202503"
sample=$(ls *_1_trimmed_paired.fastq.gz | head -1 | sed 's/_1_trimmed_paired.fastq.gz//')
metaphlan ${sample}_1_trimmed_paired.fastq.gz,${sample}_2_trimmed_paired.fastq.gz \
  --input_type fastq --nproc $SLURM_CPUS_PER_TASK \
  --db_dir "$METAPHLAN_DB" -x "$METAPHLAN_INDEX" \
  --mapout ${sample}_mapout.sam -o ${sample}_metaphlan_test.txt
  
# Test is successful: time ~46 min, memory ~34 GB.

**Substep 1.3** Running Kraken2 on all samples (array)

run_kraken_all.sbatch script

In [ ]:
#!/bin/bash
#SBATCH -J kraken_all
#SBATCH -n 1
#SBATCH --cpus-per-task=4
#SBATCH --mem=250G
#SBATCH -t 48:00:00
#SBATCH --partition=main
#SBATCH --error=/mnt/tank/scratch/ris/SRR_files/Logs/kraken_all_%j.err
#SBATCH --output=/mnt/tank/scratch/ris/SRR_files/Logs/kraken_all_%j.out

source /nfs/home/ris/miniforge3/etc/profile.d/mamba.sh
mamba activate snakemake

cd /mnt/tank/scratch/ris/SRR_files || exit 1

KRAKEN_DB="/mnt/tank/scratch/ris/databases/kraken2"

# Getting a list of all the samples
samples=($(ls *_1_trimmed_paired.fastq.gz | sed 's/_1_trimmed_paired.fastq.gz//'))

total=${#samples[@]}
echo "Samples found: $total"
echo "General task start: $(date)"

for i in "${!samples[@]}"; do
    sample=${samples[$i]}
    echo "Sample processing $((i+1))/$total: $sample"
    echo "Start: $(date)"

    if [[ ! -f "${sample}_1_trimmed_paired.fastq.gz" ]] || [[ ! -f "${sample}_2_trimmed_paired.fastq.gz" ]]; then
        echo "ERROR: missing paired files for $sample, skipping"
        continue
    fi # closing statement

    kraken2 --db "$KRAKEN_DB" \
        --threads $SLURM_CPUS_PER_TASK \
        --paired "${sample}_1_trimmed_paired.fastq.gz" "${sample}_2_trimmed_paired.fastq.gz" \
        --output "${sample}.kraken.out" \
        --report "${sample}.kraken.report" \
         --use-names

    echo "End: $(date)"
done

echo "All samples have been processed. End time: $(date)"

**Substep 1.4** Running MetaPhlAn on all samples (array)

The run_metaphls_all_array.sbatch script.

In [ ]:
#!/bin/bash
#SBATCH -J metaphlan_array
#SBATCH -a 1-57        
#SBATCH -n 1
#SBATCH --cpus-per-task=8
#SBATCH --mem=60G
#SBATCH -t 04:00:00
#SBATCH --partition=main
#SBATCH --error=/mnt/tank/scratch/ris/SRR_files/Logs/metaphlan_array_%a_%j.err
#SBATCH --output=/mnt/tank/scratch/ris/SRR_files/Logs/metaphlan_array_%a_%j.out

source /nfs/home/ris/miniforge3/etc/profile.d/mamba.sh
mamba activate snakemake

cd /mnt/tank/scratch/ris/SRR_files || exit 1

METAPHLAN_DB="/mnt/tank/scratch/ris/databases/metaphlan"
METAPHLAN_INDEX="mpa_vJan25_CHOCOPhlAnSGB_202503"

# Getting a list of samples
# mapfile -t samples < <(ls *_1_trimmed_paired.fastq.gz | sed 's/_1_trimmed_paired.fastq.gz//')
# total=${#samples[@]}

# idx=$((SLURM_ARRAY_TASK_ID - 1))
# sample=${samples[$idx]}

# echo "Sample processing $SLURM_ARRAY_TASK_ID/$total: $sample"
# echo "Start: $(date)"

# if [[ ! -f "${sample}_1_trimmed_paired.fastq.gz" ]] || [[ ! -f "${sample}_2_trimmed_paired.fastq.gz" ]]; then
#     echo "ERROR: missing paired files for $sample"
#     exit 1
# fi
# Delete the old mapout, if any (when restarting)
# rm -f "${sample}_mapout.sam"

metaphlan "${sample}_1_trimmed_paired.fastq.gz,${sample}_2_trimmed_paired.fastq.gz" \
    --input_type fastq \
    --nproc $SLURM_CPUS_PER_TASK \
    --db_dir "$METAPHLAN_DB" \
    -x "$METAPHLAN_INDEX" \
    --mapout "${sample}_mapout.sam" \
    --offline \
    -o "${sample}_metaphlan.txt"

# Deleting mapout after completion (so as not to clog the disk)
rm -f "${sample}_mapout.sam"

echo "Completed: $(date)"

**Substep 1.5.1** Assembling the Kraken2 results

A single table kraken_species_reads.csv is created from the *.kraken.report reports (rows are views, columns are samples, values are the number of reads).

The script merge_kraken.py:

In [ ]:
import pandas as pd
import glob

# Find all the report files
reports = glob.glob('*.kraken.report')
print(f"Найдено отчётов: {len(reports)}")

# Dictionary for storing data of each sample
data = {}

for rep in reports:
    # Extract the sample name from the file name (before .kraken.report)
    sample = rep.replace('.kraken.report', '')
    
    # Reading the report. It has columns: percentage, number of reads in the clade, number of reads of the taxon, rank, taxid, name
    df = pd.read_csv(rep, sep='\t', header=None,
                     names=['perc', 'clade_reads', 'taxon_reads', 'rank', 'taxid', 'name'])
    
    # Leaving only the rows with the rank 'S' (species)
    df = df[df['rank'] == 'S'].copy()
    
    # Clearing the names of unnecessary spaces
    df['name'] = df['name'].str.strip()
    
    # Setting the type name as the index
    df.set_index('name', inplace=True)
    
    # Save a column with the number of reads for this sample
    data[sample] = df['taxon_reads']

# Combine all the samples into one table. Fill in the gaps (species not found in the sample) with zeros.
merged = pd.DataFrame(data).fillna(0).astype(int)

# Saving the result
merged.to_csv('kraken_species_reads.csv')
print(f"Saved types: {merged.shape[0]}, samples: {merged.shape[1]}")

**Subset 1.5.2** Creating sample tables for Kraken2

The script split_kraken_by_sample.py:

In [ ]:
import pandas as pd
import os

df = pd.read_csv('kraken_species_reads.csv', index_col=0)
os.makedirs('kraken_by_sample', exist_ok=True)

for sample in df.columns:
    series = df[sample][df[sample] > 0]
    sample_df = pd.DataFrame({'species': series.index, 'reads': series.values})
    sample_df = sample_df.sort_values('reads', ascending=False)
    sample_df.to_csv(f'kraken_by_sample/{sample}.csv', index=False)
    print(f"Saved {sample}.csv with {len(sample_df)} views")

**Subset 1.6.1** Assembling the results of MetaPhlAn (in an exemplary manner)

Instead of combining them into one large table, we immediately use each *_metaphlan.txt creating a separate CSV with the views and their relative abundance.

The script extract_metaphlan_species.py

In [ ]:
import pandas as pd
import glob
import os

files = sorted(glob.glob('*_metaphlan.txt'))
print(f"Found files: {len(files)}")
os.makedirs('metaphlan_by_sample', exist_ok=True)

for f in files:
    sample = f.replace('_metaphlan.txt', '')
    print(f"Processing {sample}...")
    df = pd.read_csv(f, sep='\t', comment='#', header=None,
                     names=['clade_name', 'NCBI_tax_id', 'relative_abundance', 'additional_species'])
    #Leaving only the lines where there are 's__' (types)
    species = df[df['clade_name'].str.contains('s__', na=False)].copy()
    if len(species) == 0:
        print(f"There are no views in the {f} file")
        continue
    # Extract the short name of the type (the last element after the '|')
    species['species'] = species['clade_name'].apply(lambda x: x.split('|')[-1])
    result = species[['species', 'relative_abundance']].sort_values('relative_abundance', ascending=False)
    result.to_csv(f'metaphlan_by_sample/{sample}.csv', index=False)
    print(f" Saved file with {len(result)} views")

**Subset 1.6.2** Script for building a shared MetaPhlAn table

We run the following script on the server (merge_metaphlan_samples.py ):

In [ ]:
import pandas as pd
import glob
import os

# We get a list of all files with views for each sample
files = sorted(glob.glob('metaphlan_by_sample/*.csv'))
print(f"Files found: {len(files)}")

data = {}  # dictionary: {sample: DataFrame with index by type}
for f in files:
    sample = os.path.basename(f).replace('.csv', '')
    df = pd.read_csv(f)
    # Setting the view as an index
    df.set_index('species', inplace=True)
    data[sample] = df['relative_abundance']

# Combining all the samples into one table
merged = pd.DataFrame(data).fillna(0)
# Sort the rows in descending order of average abundance (optional)
merged['mean'] = merged.mean(axis=1)
merged = merged.sort_values('mean', ascending=False).drop('mean', axis=1)

# Save it in CSV with escaping (in case of commas in names)
merged.to_csv('metaphlan_abundance_all_samples.csv', quoting=1)  # quoting=1-QUOTE_ALL
print(f"Saved types: {merged.shape[0]}, samples: {merged.shape[1]}")

**Substep 1.7** Analysis locally on a computer 

For each sample, there are CSV files with the species and reads (or relative_abundance) columns

Further analysis of data is neended renemae:

**metaphlan_abundance_all_samples.csv** –> **metaphlan_step1.csv** (aviable in linked zenodo repo)

**kraken_species_reads.csv** –> **kraken_step1.csv** (aviable in linked zenodo repo)

**SraRunTable.csv** (samples metadata) is also aviable in linked zenodo repo


**Substep 1.7.1** Metadata processing

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.stats import mannwhitneyu
import numpy as np
from statsmodels.stats.multitest import multipletests

import statsmodels.api as sm
import statsmodels.formula.api as smf

import re

In [ ]:
# Load metadata
metadata = pd.read_csv('./SraRunTable.csv')

# Rename 'Run' to 'sample'
metadata.rename(columns={'Run': 'sample'}, inplace=True)

# Create groups: healthy (Fracture == 0), case (Fracture >= 1)
metadata['group'] = metadata['Fracture'].apply(lambda x: 'healthy' if x == 0 else 'case')

# Keep only required columns
metadata = metadata[['sample', 'group', 'Fracture', 'HTOT_BMD_(g/cm2)', 'age', 'BMI']]

# Check the result
print(metadata)
print(metadata['group'].value_counts())

# display(metadata)

**Substep 1.7.2** For MetaPhlAn

In [ ]:
# For MetaPhlAn
metaphlan_df = pd.read_csv('./metaphlan_step1.csv', index_col=0)
# display(metaphlan_df)

# Transpose
metaphlan_t = metaphlan_df.T.reset_index().rename(columns={'index': 'sample'})

# Merge with metadata
df_combined_met = pd.merge(metaphlan_t, metadata, on='sample', how='inner')
df_combined_met.set_index('sample', inplace=True)

# print(df_combined_met.shape)
# Combined dataframe containing for each sample (SRR index) abundances of all taxa from MetaPhlAn,
# as well as metadata columns: group, Fracture, HTOT_BMD_(g/cm2), age, BMI.

# display(df_combined_met)

**Metaphlan Top 20 taxa abundance heatmap (grouped by fracture status)**

In [ ]:
# Define taxa columns (all except metadata columns)
taxa_cols = [col for col in df_combined_met.columns if col not in ['group', 'Fracture', 'HTOT_BMD_(g/cm2)', 'age', 'BMI']]

# Top-20 taxa by mean abundance (can also use max, or other metrics)
top_taxa = df_combined_met[taxa_cols].mean().sort_values(ascending=False).head(20).index

# Matrix with only top taxa plus group column
df_top = df_combined_met[top_taxa].copy()
df_top['group'] = df_combined_met['group']

# Sort by group
df_top_sorted = df_top.sort_values('group')

data_heatmap = df_top_sorted.drop(columns=['group']).T

plt.figure(figsize=(12, 8))
sns.heatmap(data_heatmap, cmap='viridis', xticklabels=True, yticklabels=True,
            cbar_kws={'label': 'Abundance'})
plt.title('Top 20 taxa abundance (samples grouped by fracture status)')
plt.tight_layout()
plt.savefig('metaphlan_heatmap.png', dpi=150)
plt.show()

![Metaphlan top20](/images/Step1_metaphlan/Step1_metaphlan_top20taxa.png)

**PCA of microbial abundances with outlier detection and visualization (with/without outliers)**

In [ ]:
# Prepare data and run PCA
X = df_combined_met[taxa_cols].values
X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=2)
pca_result = pca.fit_transform(X_scaled)

# Plot PCA with all samples, colored by group
plt.figure(figsize=(8,6))
for group, color in zip(['healthy', 'case'], ['green', 'red']):
    idx = df_combined_met['group'] == group
    plt.scatter(pca_result[idx, 0], pca_result[idx, 1], label=group, alpha=0.7)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%})')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%})')
plt.title('PCA of microbial abundances (colored by fracture status)')
plt.legend()
plt.savefig('metaphlan_pca.png', dpi=150)
plt.show()

# Detect outliers based on extreme PC1 or PC2 values (threshold = 10)
outliers = (np.abs(pca_result[:, 0]) > 10) | (np.abs(pca_result[:, 1]) > 10)
print("Outlier samples:", df_combined_met.index[outliers].tolist())

# Optional: describe outlier abundances (commented)
# outlier_samples = df_combined_met.index[outliers]    ('SRR25006870', 'SRR25006887', 'SRR25006895', 'SRR25006907', 'SRR25006917', 'SRR25006925')
# df_combined_met.loc[outlier_samples, taxa_cols].describe()

# Plot PCA without outliers
plt.rcdefaults()
mask = ~outliers  # keep only non-outliers

plt.figure(figsize=(7,6))
for group, color in zip(['healthy', 'case'], ['green', 'red']):
    idx = (df_combined_met['group'] == group) & mask
    plt.scatter(pca_result[idx, 0], pca_result[idx, 1], label=group, c=color, alpha=0.7)

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%})', fontsize=18)
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%})', fontsize=18)
plt.title('PCA of microbial abundances (MetaPhlAn) – outliers removed', fontsize=20)
plt.legend(fontsize=14)            
plt.tick_params(axis='both', labelsize=14)  

plt.savefig('metaphlan_pca_no_outliers.png', dpi=150)
plt.show()

![Metaphlan PCA  with outlier](/images/Step1_metaphlan/Step1_metaphlan_PCA_o.png)

![Metaphlan PCA without outlier](/images/Step1_metaphlan/Step1_metaphlan_PCA_wo.png)

**Differential abundance testing (Mann-Whitney U test with FDR correction)**

For each microbial taxon, we compare relative abundances between the healthy and case groups using the non-parametric Mann-Whitney U test (two-sided). This test is suitable for non-normally distributed abundance data. To control the false discovery rate due to multiple comparisons, we adjust p-values using the Benjamini-Hochberg (FDR) procedure. Adjusted p-values (q-values) below 0.05 are considered statistically significant.

In [ ]:
taxa_cols_met = [col for col in df_combined_met.columns 
                 if col not in ['group', 'Fracture', 'HTOT_BMD_(g/cm2)', 'age', 'BMI']]

results = []
for taxon in taxa_cols_met:
    healthy_vals = df_combined_met[df_combined_met['group'] == 'healthy'][taxon]
    case_vals = df_combined_met[df_combined_met['group'] == 'case'][taxon]
    stat, p = mannwhitneyu(healthy_vals, case_vals, alternative='two-sided')
    results.append({'taxon': taxon, 'p_value': p, 'log10_p': -np.log10(p)})

results_df = pd.DataFrame(results).sort_values('p_value')
# Apply Benjamini-Hochberg (FDR) correction for multiple testing.

# Create a copy or work with index
# Extract p-values, keeping indices (before dropna)
non_nan_mask = results_df['p_value'].notna()
pvals = results_df.loc[non_nan_mask, 'p_value'].values
indices = results_df.loc[non_nan_mask].index

# FDR correction
reject, pvals_corrected, _, _ = multipletests(pvals, alpha=0.05, method='fdr_bh')

results_df['q_value'] = np.nan
results_df.loc[indices, 'q_value'] = pvals_corrected

# Sort by q_value and show top 10
results_df.sort_values('q_value').head(10)

![Metaphlan p-value](/images/Step1_metaphlan/Step1_metaphlan_corpval.png)

**Substep 1.7.3** For Kraken2

In [ ]:
# For Kraken2
kraken_df = pd.read_csv('./kraken_step1.csv', index_col=0)
# display(kraken_df)

# Transpose
kraken_t = kraken_df.T.reset_index().rename(columns={'index': 'sample'})

# Merge with metadata
df_combined_kra = pd.merge(kraken_t, metadata, on='sample', how='inner')
df_combined_kra.set_index('sample', inplace=True)

# print(df_combined_kra.shape)
# Combined dataframe containing for each sample (SRR index) abundances of all species from Kraken2,
# as well as metadata columns: group, Fracture, HTOT_BMD_(g/cm2), age, BMI.

# display(df_combined_kra)

**Kraken2 Top 20 taxa abundance heatmap (grouped by fracture status)**

In [ ]:
# Define taxa columns (all except metadata columns)
taxa_cols = [col for col in df_combined_kra.columns 
             if col not in ['group', 'Fracture', 'HTOT_BMD_(g/cm2)', 'age', 'BMI']]

# Top-20 taxa by mean abundance
top_taxa = df_combined_kra[taxa_cols].mean().sort_values(ascending=False).head(20).index

# Matrix with only top taxa plus group column
df_top = df_combined_kra[top_taxa].copy()
df_top['group'] = df_combined_kra['group']

# Sort by group
df_top_sorted = df_top.sort_values('group')

# Data for heatmap (without group column)
data_heatmap = df_top_sorted.drop(columns=['group']).T

plt.figure(figsize=(12, 8))
sns.heatmap(data_heatmap, cmap='viridis', xticklabels=True, yticklabels=True,
            cbar_kws={'label': 'Abundance'})
plt.title('Kraken2: Top 20 species abundance (samples grouped by fracture status)')
plt.tight_layout()
plt.savefig('kraken_heatmap.png', dpi=150)
plt.show()

![Kraken top20](/images/Step1_kraken/Step1_kraken_top20taxa.png)

**PCA of microbial abundances with outlier detection and visualization (with/without outliers)**

In [ ]:
# PCA with all samples
X = df_combined_kra[taxa_cols].values
X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=2)
pca_result = pca.fit_transform(X_scaled)

plt.figure(figsize=(8,6))
for group, color in zip(['healthy', 'case'], ['blue', 'red']):
    idx = df_combined_kra['group'] == group
    plt.scatter(pca_result[idx, 0], pca_result[idx, 1], label=group, alpha=0.7)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%})')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%})')
plt.title('Kraken2: PCA of species abundances (colored by fracture status)')
plt.legend()
plt.savefig('kraken_pca.png', dpi=150)
plt.show()

# Detect outliers based on extreme PC1 or PC2 values (threshold = 150)
outliers_kra = (np.abs(pca_result[:, 0]) > 150) | (np.abs(pca_result[:, 1]) > 150)
print("Outlier samples (Kraken2):", df_combined_kra.index[outliers_kra].tolist())

# Optional: show abundance for outlier samples
# outlier_samples = df_combined_kra.index[outliers_kra]
# df_combined_kra.loc[outlier_samples, taxa_cols].describe()

# Plot PCA without outliers
mask = ~outliers_kra  # keep only non-outliers

plt.figure(figsize=(8,6))
for group, color in zip(['healthy', 'case'], ['blue', 'red']):
    idx = (df_combined_kra['group'] == group) & mask
    plt.scatter(pca_result[idx, 0], pca_result[idx, 1], label=group, alpha=0.7)

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%})')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%})')
plt.title('Kraken2: PCA of species abundances (outliers removed)')
plt.legend()
plt.tight_layout()
plt.savefig('kraken_pca_no_outliers.png', dpi=150)
plt.show()


![Kraken PCA  with outlier](/images/Step1_kraken/Step1_kraken_PCA_o.png)

![Kraken PCA without outlier](/images/Step1_kraken/Step1_kraken_PCA_wo.png)

**Differential abundance testing (Mann-Whitney U test with FDR correction)**

In [ ]:
taxa_cols_kra = [col for col in df_combined_met.columns 
                 if col not in ['group', 'Fracture', 'HTOT_BMD_(g/cm2)', 'age', 'BMI']]
results = []
for taxon in taxa_cols:
    healthy_vals = df_combined_kra[df_combined_kra['group'] == 'healthy'][taxon]
    case_vals = df_combined_kra[df_combined_kra['group'] == 'case'][taxon]
    stat, p = mannwhitneyu(healthy_vals, case_vals, alternative='two-sided')
    results.append({'taxon': taxon, 'p_value': p, 'log10_p': -np.log10(p)})

# Sort by raw p-value
results_df_kra = pd.DataFrame(results).sort_values('p_value')
print("Top 10 by raw p-value:")
print(results_df_kra.head(10))

# FDR correction (Benjamini-Hochberg)
non_nan_mask = results_df_kra['p_value'].notna()
pvals = results_df_kra.loc[non_nan_mask, 'p_value'].values
indices = results_df_kra.loc[non_nan_mask].index

reject, pvals_corrected, _, _ = multipletests(pvals, alpha=0.05, method='fdr_bh')

results_df_kra['q_value'] = np.nan
results_df_kra.loc[indices, 'q_value'] = pvals_corrected

# Sort by q_value and display top 10
print("\nTop 10 by FDR-adjusted q-value:")
print(results_df_kra.sort_values('q_value').head(10))

![Kraken p-value](/images/Step1_kraken/Step1_kraken_corpval.png)

**Substep 1.8** Alpha diversity (Shannon index) comparison between healthy and case groups across profiling methods

**Substep 1.8.1** for Fracture

In [ ]:
# metadata['group'] = metadata['Fracture'].apply(lambda x: 'healthy' if x == 0 else 'case')
metadata['fracture_group'] = metadata['group']  # 'healthy' or 'case'

kraken_df = pd.read_csv('./kraken_step1.csv', index_col=0)
kraken_t = kraken_df.T.reset_index().rename(columns={'index': 'sample'})
df_combined_kra = pd.merge(kraken_t, metadata, on='sample', how='inner')
df_combined_kra.set_index('sample', inplace=True)

metaphlan_df = pd.read_csv('./metaphlan_step1.csv', index_col=0)
metaphlan_t = metaphlan_df.T.reset_index().rename(columns={'index': 'sample'})
df_combined_met = pd.merge(metaphlan_t, metadata, on='sample', how='inner')
df_combined_met.set_index('sample', inplace=True)

def shannon_index(df, taxa_cols):
    """Calculate Shannon diversity index for each row (sample)."""
    abundances = df[taxa_cols].values
    row_sums = abundances.sum(axis=1, keepdims=True)
    abundances = abundances / row_sums
    abundances = np.where(abundances == 0, 1e-12, abundances)
    shannon = -np.sum(abundances * np.log(abundances), axis=1)
    return shannon

# Compute Shannon index for Kraken2
taxa_cols_kra = [col for col in df_combined_kra.columns 
                 if col not in ['group', 'fracture_group', 'Fracture', 'HTOT_BMD_(g/cm2)', 'age', 'BMI']]
df_combined_kra['shannon'] = shannon_index(df_combined_kra, taxa_cols_kra)

# Compute Shannon index for MetaPhlAn
taxa_cols_met = [col for col in df_combined_met.columns 
                 if col not in ['group', 'fracture_group', 'Fracture', 'HTOT_BMD_(g/cm2)', 'age', 'BMI']]
df_combined_met['shannon'] = shannon_index(df_combined_met, taxa_cols_met)

alpha_data_fracture = pd.DataFrame({
    'shannon': pd.concat([df_combined_kra['shannon'], df_combined_met['shannon']], ignore_index=True),
    'fracture_group': pd.concat([df_combined_kra['fracture_group'], df_combined_met['fracture_group']], ignore_index=True),
    'method': ['Kraken2'] * len(df_combined_kra) + ['MetaPhlAn'] * len(df_combined_met)
})

# 8. Mann-Whitney U test for each method
def get_u_and_p_fracture(data, method_name):
    subset = data[data['method'] == method_name]
    healthy = subset[subset['fracture_group'] == 'healthy']['shannon']
    case = subset[subset['fracture_group'] == 'case']['shannon']
    u_stat, p_val = mannwhitneyu(healthy, case, alternative='two-sided')
    return u_stat, p_val

u_kra_fx, p_kra_fx = get_u_and_p_fracture(alpha_data_fracture, 'Kraken2')
u_met_fx, p_met_fx = get_u_and_p_fracture(alpha_data_fracture, 'MetaPhlAn')

print("\nMann-Whitney U test results (healthy vs case based on fracture):")
print(f'Kraken2:     U = {u_kra_fx:.1f}, p = {p_kra_fx:.4f}')
print(f'MetaPhlAn:   U = {u_met_fx:.1f}, p = {p_met_fx:.4f}')

fracture_colors = {'healthy': 'green', 'case': 'red'}

plt.figure(figsize=(8,5))
sns.boxplot(
    x='method', 
    y='shannon', 
    hue='fracture_group', 
    data=alpha_data_fracture, 
    palette=fracture_colors
)

plt.title('Alpha diversity (Shannon index) by fracture status and method', fontsize=18)
plt.xlabel('Method', fontsize=16) 
plt.ylabel('Shannon index', fontsize=16)
plt.legend(title='Fracture status', fontsize=12)
plt.tick_params(axis='both', labelsize=14)

plt.tight_layout()
plt.savefig('alpha_diversity_fracture_comparison.png', dpi=150)
plt.show()

![BMD_alfa_div](/images/Fracture/Fracture_alpha_diversity.png)

**Substep 1.8.2** for BMD

In [ ]:
metadata['bmd_group'] = metadata['sample'].map(bmd_dict) # bmd_dict is grouping based on Step 0/substep 0.6 ('SRR25006867': 'normal', ....)
print("BMD group counts:")
print(metadata['bmd_group'].value_counts(dropna=False))

kraken_df = pd.read_csv('./kraken_step1.csv', index_col=0)
kraken_t = kraken_df.T.reset_index().rename(columns={'index': 'sample'})
df_combined_kra = pd.merge(kraken_t, metadata, on='sample', how='inner')
df_combined_kra.set_index('sample', inplace=True)

metaphlan_df = pd.read_csv('./metaphlan_step1.csv', index_col=0)
metaphlan_t = metaphlan_df.T.reset_index().rename(columns={'index': 'sample'})
df_combined_met = pd.merge(metaphlan_t, metadata, on='sample', how='inner')
df_combined_met.set_index('sample', inplace=True)

def shannon_index(df, taxa_cols):
    """Calculate Shannon diversity index for each row (sample)."""
    abundances = df[taxa_cols].values
    # Normalize to relative abundances (if not already)
    row_sums = abundances.sum(axis=1, keepdims=True)
    abundances = abundances / row_sums
    # Replace zeros with a very small number to avoid log(0)
    abundances = np.where(abundances == 0, 1e-12, abundances)
    shannon = -np.sum(abundances * np.log(abundances), axis=1)
    return shannon

# Compute Shannon index for Kraken2
taxa_cols_kra = [col for col in df_combined_kra.columns 
                 if col not in ['group', 'Fracture', 'HTOT_BMD_(g/cm2)', 'age', 'BMI', 'bmd_group']]
df_combined_kra['shannon'] = shannon_index(df_combined_kra, taxa_cols_kra)

# Compute Shannon index for MetaPhlAn
taxa_cols_met = [col for col in df_combined_met.columns 
                 if col not in ['group', 'Fracture', 'HTOT_BMD_(g/cm2)', 'age', 'BMI', 'bmd_group']]
df_combined_met['shannon'] = shannon_index(df_combined_met, taxa_cols_met)

alpha_data_bmd = pd.DataFrame({
    'shannon': pd.concat([df_combined_kra['shannon'], df_combined_met['shannon']], ignore_index=True),
    'bmd_group': pd.concat([df_combined_kra['bmd_group'], df_combined_met['bmd_group']], ignore_index=True),
    'method': ['Kraken2'] * len(df_combined_kra) + ['MetaPhlAn'] * len(df_combined_met)
})

# Mann-Whitney U test for each method
def get_u_and_p_bmd(data, method_name):
    subset = data[data['method'] == method_name]
    normal = subset[subset['bmd_group'] == 'normal']['shannon']
    low = subset[subset['bmd_group'] == 'low']['shannon']
    u_stat, p_val = mannwhitneyu(normal, low, alternative='two-sided')
    return u_stat, p_val

u_kra_bmd, p_kra_bmd = get_u_and_p_bmd(alpha_data_bmd, 'Kraken2')
u_met_bmd, p_met_bmd = get_u_and_p_bmd(alpha_data_bmd, 'MetaPhlAn')

print("\nMann-Whitney U test results (normal vs low BMD):")
print(f'Kraken2:     U = {u_kra_bmd:.1f}, p = {p_kra_bmd:.4f}')
print(f'MetaPhlAn:   U = {u_met_bmd:.1f}, p = {p_met_bmd:.4f}')

bmd_colors = {'normal': 'blue', 'low': 'orange'}

plt.figure(figsize=(8,5))
sns.boxplot(
    x='method', 
    y='shannon', 
    hue='bmd_group', 
    data=alpha_data_bmd, 
    palette=bmd_colors
)

plt.title('Alpha diversity (Shannon index) by BMD group and method', fontsize=18)
plt.xlabel('Method', fontsize=16) 
plt.ylabel('Shannon index', fontsize=16)
plt.legend(title='BMD status', fontsize=12)
plt.tick_params(axis='both', labelsize=14)

plt.tight_layout()
plt.savefig('alpha_diversity_BMD_comparison.png', dpi=150)
plt.show()

![BMD_alfa_div](/images/BMD/BMD_alpha_diversity.png)